# Entrega 5: obtenção, limpeza e tratamento dos dados

**TPAE II (PSAE00320) · Equipe 01** · Ana Beatriz Laurentino, Arthur Almeida e Gabriel Pinheiro · Orientador: Prof. Alessio
Demandante: CORECON-PB · Repositório: <https://github.com/laurentinoanabia/TPAEII_equipe01>

## 1. Objetivo, fontes e produtos gerados

**Objetivo.** Produzir bases analíticas consistentes, preservando os dados originais e registrando todas as transformações. O notebook executa do início ao fim, sem edição manual.

**Duas bases, com situações diferentes. Leia antes de usar os resultados:**

| Base | Fonte (inventário) | Situação em 29/10/2026 | O que este notebook produz |
|---|---|---|---|
| **Glossário** | F2 · `data/glossario.json` | **Dado real**: 6 verbetes de rascunho escritos pela equipe | Tabelas tratadas em `data/processed/` |
| **Questionário** | F1 · `data/raw/respostas_questionario.csv` | **Ainda não aplicado.** Sem o arquivo real, o notebook gera uma base **SINTÉTICA** com defeitos plantados, só para testar as regras de limpeza | Em `outputs/sintetico_teste/`. **Não é resultado de pesquisa.** Quando o arquivo real existir, o mesmo notebook o trata e grava as tabelas agregadas em `data/processed/` |

**Produtos**

| Arquivo | Conteúdo |
|---|---|
| `data/processed/glossario_verbetes.csv` | Um verbete por linha, com variáveis derivadas |
| `data/processed/glossario_relacoes.csv` | Uma relação "Veja também" por linha |
| `data/processed/glossario_cobertura_questionario.csv` | Verbetes que cada item do questionário exige, e se existem |
| `outputs/relatorio_qualidade.csv` | Tabela Variável/base · Problema · Decisão · Impacto/validação |
| `outputs/funil_contagens.csv` | Contagem de linhas antes e depois de cada etapa |
| `outputs/testes_integridade.csv` | Resultado dos testes automáticos |
| `outputs/livro_de_codigos.csv` | Códigos válidos de cada variável do questionário |
| `outputs/sintetico_teste/*.csv` | Tabelas do teste com dados sintéticos |

## 2. Importação de bibliotecas e configuração reproduzível

In [1]:
import hashlib, importlib.util, platform, random, re, sys, unicodedata
import json
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

SEMENTE = 20261029                       # fixa o teste sintético: mesma execução, mesmos números
MIN_CELULA = 5                           # células com menos respondentes são suprimidas

def achar_raiz():
    """Sobe a partir da pasta atual até achar a raiz do repositório (README.md + data/)."""
    p = Path.cwd().resolve()
    for d in [p, *p.parents]:
        if (d / "README.md").exists() and (d / "data").is_dir():
            return d
    raise FileNotFoundError("Raiz do repositório não encontrada. Rode o notebook de dentro do repositório.")

RAIZ = achar_raiz()
GLOSSARIO = RAIZ / "data" / "glossario.json"
RESPOSTAS_REAIS = RAIZ / "data" / "raw" / "respostas_questionario.csv"
PROCESSADOS = RAIZ / "data" / "processed"
SAIDAS = RAIZ / "outputs"
SINTETICO = SAIDAS / "sintetico_teste"
for pasta in (PROCESSADOS, SAIDAS, SINTETICO):
    pasta.mkdir(parents=True, exist_ok=True)

print("Execução (UTC):", datetime.now(timezone.utc).isoformat(timespec="seconds"))
print("Python", platform.python_version(), "| pandas", pd.__version__, "| semente", SEMENTE)
print("Raiz do repositório:", RAIZ.name)

Execução (UTC): 2026-10-27T13:48:26+00:00
Python 3.13.17 | pandas 3.0.5 | semente 20261029
Raiz do repositório: tpaeii_equipe01


In [2]:
# Registros que alimentam o relatório de qualidade e os testes. Tudo o que o notebook decide passa por aqui.
relatorio, funil, testes = [], [], []

def registrar(base, problema, decisao, impacto, n=None, status=None):
    """n = nº de ocorrências do problema (0 = verificado e sem ocorrência; None = só informativo)."""
    status = status or ("informativo" if n is None else "sem ocorrência" if n == 0 else "tratado")
    relatorio.append({"base_variavel": base, "status": status, "ocorrencias": n, "problema_encontrado": problema,
                      "decisao_aplicada": decisao, "impacto_validacao": impacto})

def contar(base, etapa, n):
    funil.append({"base": base, "etapa": etapa, "linhas": int(n)})

def testar(nome, condicao, detalhe=""):
    testes.append({"teste": nome, "resultado": "OK" if condicao else "FALHOU", "detalhe": detalhe})
    assert condicao, f"Teste falhou: {nome} {detalhe}"

# ---- funções de limpeza de texto
def sem_acento(s):
    return "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")

def slug(s):
    return re.sub(r"[^a-z0-9]+", "-", sem_acento(str(s)).lower()).strip("-")

def limpa_texto(x):
    """Normaliza Unicode (NFC), colapsa espaços e apara. Vazio vira NA."""
    if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA:
        return pd.NA
    t = re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(x))).strip()
    return t if t else pd.NA

def norm_cod(x):
    """Código de resposta: sem acento, minúsculo, espaços/hífens viram '_'."""
    t = limpa_texto(x)
    if t is pd.NA:
        return pd.NA
    return re.sub(r"[\s\-/]+", "_", sem_acento(t).lower())

# ---- funções de agregação já testadas em notebooks/02_tratamento_questionario.py (reuso, sem copiar código)
spec = importlib.util.spec_from_file_location("trat", RAIZ / "notebooks" / "02_tratamento_questionario.py")
M = importlib.util.module_from_spec(spec); spec.loader.exec_module(M)
print("Reutilizando de 02_tratamento_questionario.py:", [f for f in ("frequencias", "cruzar", "medias", "nivel") if hasattr(M, f)])

Reutilizando de 02_tratamento_questionario.py: ['frequencias', 'cruzar', 'medias', 'nivel']


In [3]:
# ---- Livro de códigos: o "contrato" entre o formulário e o tratamento.
# O formulário real deve exportar exatamente estes códigos (ou o mapeamento é feito antes do tratamento).
def faixa(*pares): return dict(pares)
DOMINIOS = {
  "consentimento": {"aceito": "Aceita participar", "nao_aceito": "Não aceita participar"},
  "a2_periodo": {"1_4": "1º ao 4º período", "5_8": "5º ao 8º período", "9_mais": "9º período ou mais"},
  "a3_idade": {"18_20": "18 a 20 anos", "21_23": "21 a 23 anos", "24_26": "24 a 26 anos", "27_mais": "27 anos ou mais"},
  "a4_ocupacao": {"so_estuda": "Só estuda", "estuda_trabalha": "Estuda e trabalha ou estagia", "outra": "Outra situação"},
  "a5_renda_propria": {"nenhuma": "Nenhuma", "ate_1": "Até 1 salário mínimo", "1_2": "Mais de 1 até 2", "mais_2": "Mais de 2", "pnr": "Prefiro não responder"},
  "b1_reserva_def": {"a": "Dinheiro guardado para comprar algo desejado", "b": "Dinheiro de fácil acesso para imprevistos ou perda de renda (correta)",
                     "c": "Dinheiro investido em ações para ganhar o máximo", "d": "Limite do cartão ou cheque especial", "e": "Não sei"},
  "b2_reserva_meses": {"menos_1": "Menos de 1 mês", "1_3": "De 1 a 3 meses", "3_6": "De 3 a 6 meses", "mais_6": "Mais de 6 meses", "nao_sei": "Não sei"},
  "b3_inflacao": {"a": "Mais do que hoje", "b": "Exatamente o mesmo", "c": "Menos do que hoje (correta)", "d": "Não sei"},
  "b4_juros_compostos": {"a": "R$ 1.200", "b": "R$ 1.210 (correta)", "c": "R$ 1.100", "d": "Não sei"},
  "b5_diversificacao": {"a": "Risco maior (correta)", "b": "Risco menor", "c": "O mesmo risco", "d": "Não sei"},
  "b6_autoavaliacao": {str(i): f"{i} de 5" for i in range(1, 6)},
  "c1_orcamento": {"nunca": "Nunca", "as_vezes": "Às vezes", "sempre": "Sempre"},
  "c2_tem_reserva": {"sim": "Sim", "parcial": "Parcialmente", "nao": "Não"},
  "c3_reserva_cobertura": {"menos_1": "Menos de 1 mês", "1_3": "De 1 a 3 meses", "3_6": "De 3 a 6 meses", "mais_6": "Mais de 6 meses"},
  "c4_onde_guarda": {"poupanca": "Poupança", "renda_fixa": "Renda fixa", "conta_corrente": "Conta corrente", "dinheiro_casa": "Dinheiro em casa", "outro": "Outro"},
  "c5_dividas": {"cartao": "Cartão de crédito", "emprestimo": "Empréstimo ou financiamento", "atrasadas": "Parcelas atrasadas", "nenhuma": "Nenhuma", "pnr": "Prefiro não responder"},
  "c6_poupa_regular": {"sim": "Sim", "as_vezes": "Às vezes", "nao": "Não"},
  "d1_renda_familiar": {"ate_1": "Até 1 salário mínimo", "1_3": "Mais de 1 até 3", "3_6": "Mais de 3 até 6", "mais_6": "Mais de 6", "nao_sei": "Não sei", "pnr": "Prefiro não responder"},
  "d2_fam_conversa": {"nunca": "Nunca", "as_vezes": "Às vezes", "frequente": "Frequentemente"},
  "d3_fam_orcamento": {"sim": "Sim", "nao": "Não", "nao_sei": "Não sei"},
  "d4_fam_reserva": {"sim": "Sim", "nao": "Não", "nao_sei": "Não sei"},
  "d5_fam_dividas": {"sim": "Sim", "nao": "Não", "nao_sei": "Não sei", "pnr": "Prefiro não responder"},
  "d6_fam_conhecimento": {**{str(i): f"{i} de 5" for i in range(1, 6)}, "nao_sei": "Não sei"},
  "d7_educ_recebida": {"escola": "Escola", "familia": "Família", "curso": "Curso ou faculdade", "internet": "Internet e redes sociais", "nunca": "Nunca tive"},
  "e1_temas_interesse": {"orcamento": "Orçamento", "reserva": "Reserva de emergência", "credito_juros": "Crédito e juros", "dividas": "Dívidas e renegociação",
                         "investimentos": "Investimentos", "inflacao_selic": "Inflação e Selic", "impostos": "Impostos", "outro": "Outro"},
  "e2_usaria_glossario": {"sim": "Sim", "talvez": "Talvez", "nao": "Não"},
}
MULTIPLAS = set(M.MULTIPLA)
livro = pd.DataFrame([{"variavel": v, "tipo": "multipla" if v in MULTIPLAS else "unica", "codigo": c, "rotulo": r}
                      for v, d in DOMINIOS.items() for c, r in d.items()])
print(f"{livro.variavel.nunique()} variáveis com domínio fechado, {len(livro)} códigos. a1_curso é texto livre (lista de cursos a definir).")
livro.head(6)

26 variáveis com domínio fechado, 108 códigos. a1_curso é texto livre (lista de cursos a definir).


,variavel,tipo,codigo,rotulo
0,consentimento,unica,aceito,Aceita participar
1,consentimento,unica,nao_aceito,Não aceita participar
2,a2_periodo,unica,1_4,1º ao 4º período
3,a2_periodo,unica,5_8,5º ao 8º período
4,a2_periodo,unica,9_mais,9º período ou mais
5,a3_idade,unica,18_20,18 a 20 anos


## 3. Aquisição ou leitura dos dados originais

**Preservação.** Os dados originais nunca são editados: os arquivos são lidos como bytes, o `sha256` é registrado e conferido de novo no fim. O glossário é um arquivo versionado no Git (recuperável por qualquer commit). As respostas reais do questionário ficam em `data/raw/`, que o `.gitignore` bloqueia; o procedimento para obtê-las novamente está em `data/raw/README.md` (exportar do formulário).

In [4]:
# ---- 3.1 Glossário (dado real)
bytes_glossario = GLOSSARIO.read_bytes()
HASH_GLOSSARIO = hashlib.sha256(bytes_glossario).hexdigest()
bruto_glossario = json.loads(bytes_glossario.decode("utf-8"))
aquisicao = [{"fonte": "F2 glossario.json", "arquivo": str(GLOSSARIO.relative_to(RAIZ)), "tamanho_bytes": len(bytes_glossario),
              "sha256": HASH_GLOSSARIO, "parametros": "leitura integral, sem filtros", "origem": "REAL"}]
print("Glossário lido:", len(bruto_glossario), "verbetes | sha256", HASH_GLOSSARIO[:16], "...")

Glossário lido: 6 verbetes | sha256 688481a2ba274403 ...


In [5]:
# ---- 3.2 Questionário: arquivo real, se existir; senão, base SINTÉTICA para testar as regras
def gerar_sintetico(n=60, semente=SEMENTE):
    """Base sintética com defeitos plantados (duplicatas, códigos inválidos, inconsistências, ausências, variantes de grafia).
    NÃO representa pessoas nem resultados: serve só para provar que as regras de limpeza funcionam."""
    rng = random.Random(semente)
    cod = lambda v: sorted(DOMINIOS[v])
    certo = {"b1_reserva_def": "b", "b3_inflacao": "c", "b4_juros_compostos": "b", "b5_diversificacao": "a"}
    unicas = ["a2_periodo", "a3_idade", "a4_ocupacao", "a5_renda_propria", "b2_reserva_meses", "b6_autoavaliacao", "c1_orcamento",
              "c2_tem_reserva", "c6_poupa_regular", "d1_renda_familiar", "d2_fam_conversa", "d3_fam_orcamento", "d4_fam_reserva",
              "d5_fam_dividas", "d6_fam_conhecimento", "e2_usaria_glossario"]
    linhas = []
    for i in range(n):
        r = {"Carimbo de data/hora": f"2026-11-{17 + i % 14:02d} {8 + i % 12:02d}:{(i * 7) % 60:02d}",
             "Nome": f"pessoa_sintetica_{i + 1:03d}", "consentimento": "aceito",
             "a1_curso": rng.choice(["Economia", "Economia", "Direito", "Engenharia", "Administração"])}
        for v in unicas:
            r[v] = rng.choice(cod(v))
        for v, ok in certo.items():   # acerta com probabilidade ~ 4/(4+k)
            r[v] = rng.choices(cod(v), weights=[4 if c == ok else 1 for c in cod(v)])[0]
        if r["c2_tem_reserva"] != "nao":
            r["c3_reserva_cobertura"] = rng.choice(cod("c3_reserva_cobertura"))
            r["c4_onde_guarda"] = ";".join(rng.sample(cod("c4_onde_guarda"), rng.choice([1, 2])))
        else:
            r["c3_reserva_cobertura"] = r["c4_onde_guarda"] = ""
        r["c5_dividas"] = ";".join(rng.sample(cod("c5_dividas"), rng.choice([1, 2])))
        r["d7_educ_recebida"] = ";".join(rng.sample(cod("d7_educ_recebida"), rng.choice([1, 2])))
        r["e1_temas_interesse"] = ";".join(rng.sample(cod("e1_temas_interesse"), rng.choice([1, 2, 3])))
        r["e3_termo_dificil"] = rng.choice(["", "juros compostos", "selic", ""])
        linhas.append(r)
    df = pd.DataFrame(linhas)

    def variante(v):                   # grafias diferentes do mesmo código
        if v == "nao": return rng.choice(["não", "NÃO", " nao "])
        if v == "as_vezes": return rng.choice(["às vezes", "As Vezes", "as-vezes"])
        return rng.choice([v.upper(), f"  {v} ", v.replace("_", " "), v.replace("_", "-")])
    for _ in range(25):
        i, c = rng.randrange(n), rng.choice(["c1_orcamento", "c2_tem_reserva", "c6_poupa_regular", "d2_fam_conversa", "d3_fam_orcamento", "d4_fam_reserva", "a2_periodo", "d1_renda_familiar"])
        if df.at[i, c]: df.at[i, c] = variante(df.at[i, c])
    for i, c, v in [(3, "b1_reserva_def", "z"), (11, "b6_autoavaliacao", "7"), (19, "c1_orcamento", "talvez"),
                    (27, "d2_fam_conversa", "sempre"), (41, "a3_idade", "17_ou_menos"), (52, "b3_inflacao", "x")]:
        df.at[i, c] = v                                              # códigos inválidos
    for i in [5, 13, 22, 34, 47, 55]:                                # c3/c4 preenchidos sem ter reserva
        df.loc[i, ["c2_tem_reserva", "c3_reserva_cobertura", "c4_onde_guarda"]] = ["nao", "1_3", "poupanca"]
    for _ in range(10):                                              # ausências
        df.at[rng.randrange(n), rng.choice(["c1_orcamento", "d1_renda_familiar", "d3_fam_orcamento", "b2_reserva_meses"])] = ""
    for i in [8, 30, 44, 58]:                                        # mais de 3 temas
        df.at[i, "e1_temas_interesse"] = "orcamento;reserva;credito_juros;dividas"
    for i in [2, 16, 37, 49]:                                        # sem consentimento
        df.at[i, "consentimento"] = "nao_aceito"
    return pd.concat([df, df.iloc[[0, 9, 25]]], ignore_index=True)   # 3 envios duplicados

if RESPOSTAS_REAIS.exists():
    ORIGEM_Q = "REAL"
    questionario_bruto = pd.read_csv(RESPOSTAS_REAIS, dtype=str, keep_default_na=False, na_values=[""])
    arq_bytes = RESPOSTAS_REAIS.read_bytes()
    aquisicao.append({"fonte": "F1 questionario", "arquivo": "data/raw/respostas_questionario.csv", "tamanho_bytes": len(arq_bytes),
                      "sha256": hashlib.sha256(arq_bytes).hexdigest(), "parametros": "exportação integral do formulário", "origem": "REAL"})
else:
    ORIGEM_Q = "SINTETICO"
    questionario_bruto = gerar_sintetico()
    aquisicao.append({"fonte": "F1 questionario", "arquivo": "(gerado em memória)", "tamanho_bytes": 0, "sha256": "",
                      "parametros": f"gerar_sintetico(n=60, semente={SEMENTE})", "origem": "SINTETICO"})
print(f"Questionário: origem = {ORIGEM_Q} | {questionario_bruto.shape[0]} linhas x {questionario_bruto.shape[1]} colunas")
if ORIGEM_Q == "SINTETICO":
    print("ATENÇÃO: não existe data/raw/respostas_questionario.csv. Tudo abaixo sobre o questionário é TESTE com dados sintéticos.")
pd.DataFrame(aquisicao)

Questionário: origem = SINTETICO | 63 linhas x 30 colunas
ATENÇÃO: não existe data/raw/respostas_questionario.csv. Tudo abaixo sobre o questionário é TESTE com dados sintéticos.


,fonte,arquivo,tamanho_bytes,sha256,parametros,origem
0,F2 glossario.json,data/glossario.json,2835,688481a2ba274403d320ee3fc1941f00e9bde594e22c92...,"leitura integral, sem filtros",REAL
1,F1 questionario,(gerado em memória),0,,"gerar_sintetico(n=60, semente=20261029)",SINTETICO


## 4. Diagnóstico inicial de estrutura e qualidade

In [6]:
def diagnostico(df, nome):
    em_branco = df.isna() | df.astype(str).apply(lambda s: s.str.strip() == "")
    d = pd.DataFrame({"tipo": df.dtypes.astype(str), "ausentes_ou_em_branco": em_branco.sum(), "distintos": df.astype(str).nunique()})
    print(f"[{nome}] {df.shape[0]} linhas x {df.shape[1]} colunas | linhas duplicadas: {int(df.astype(str).duplicated().sum())}")
    return d

vb_bruto = pd.DataFrame(bruto_glossario)
print("Colunas esperadas do glossário ausentes:", sorted(set(["termo","tema","definicao","exemplo","importa","relacionados","fonte"]) - set(vb_bruto.columns)))
display(diagnostico(vb_bruto.astype({"relacionados": str}) if "relacionados" in vb_bruto else vb_bruto, "glossário"))
display(diagnostico(questionario_bruto, f"questionário ({ORIGEM_Q})").head(12))

Colunas esperadas do glossário ausentes: ['fonte']
[glossário] 6 linhas x 6 colunas | linhas duplicadas: 0


,tipo,ausentes_ou_em_branco,distintos
termo,str,0,6
tema,str,0,3
definicao,str,0,6
exemplo,str,0,6
importa,str,0,6
relacionados,str,0,6


[questionário (SINTETICO)] 63 linhas x 30 colunas | linhas duplicadas: 3


,tipo,ausentes_ou_em_branco,distintos
Carimbo de data/hora,str,0,60
Nome,str,0,60
consentimento,str,0,2
a1_curso,str,0,4
a2_periodo,str,0,4
a3_idade,str,0,5
a4_ocupacao,str,0,3
a5_renda_propria,str,0,5
b2_reserva_meses,str,4,6
b6_autoavaliacao,str,0,6


## 5. Tratamentos organizados em etapas nomeadas

### 5.1 Glossário (dado real)
Etapas: **G1** normalizar texto · **G2** chave e duplicidades · **G3** relações em tabela própria · **G4** variáveis derivadas · **G5** cobertura do questionário.

In [7]:
# ---------- G1: normalizar texto
COLS_TEXTO = ["termo", "tema", "definicao", "exemplo", "importa", "fonte"]
vb = vb_bruto.copy()
for c in COLS_TEXTO:
    if c not in vb.columns:
        vb[c] = pd.NA
alterados = 0
for c in COLS_TEXTO:
    novo = vb[c].map(limpa_texto)
    alterados += int(((vb[c].astype("string") != novo.astype("string")) & vb[c].notna()).sum())
    vb[c] = novo
contar("glossario", "bruto", len(vb_bruto))
registrar("glossario · textos", "Espaços extras ou formas Unicode diferentes nos campos de texto",
          "Normalizar para NFC, colapsar espaços e aparar as pontas",
          f"{alterados} células alteradas de {len(vb) * len(COLS_TEXTO)}; nenhuma linha removida", n=alterados)

# ---------- G2: chave e duplicidades
vb["slug"] = vb["termo"].map(slug)
dup = int(vb["slug"].duplicated().sum())
sem_termo = int(vb["termo"].isna().sum())
registrar("glossario · termo/slug", "Termos duplicados (mesmo identificador no site) ou sem nome",
          "Verificar; remoção só se houver duplicata exata" if dup else "Nenhuma ação necessária",
          f"{dup} duplicados e {sem_termo} sem termo em {len(vb)} verbetes", n=dup + sem_termo)
testar("Glossário: chave (slug) única e não vazia", dup == 0 and sem_termo == 0)
contar("glossario", "após G2 (duplicidades)", len(vb))

In [8]:
# ---------- G3: relações "Veja também" em tabela própria (uma relação por linha)
if "relacionados" not in vb.columns:
    vb["relacionados"] = [[] for _ in range(len(vb))]
vb["relacionados"] = vb["relacionados"].map(lambda x: [limpa_texto(r) for r in x] if isinstance(x, list) else [])
rel = (vb[["slug", "termo", "relacionados"]].rename(columns={"slug": "slug_origem", "termo": "termo_origem"})
         .explode("relacionados").dropna(subset=["relacionados"]).rename(columns={"relacionados": "termo_destino"}))
rel["slug_destino"] = rel["termo_destino"].map(slug)
slugs = set(vb["slug"])
pares = set(zip(rel["slug_origem"], rel["slug_destino"]))
rel["destino_existe"] = rel["slug_destino"].isin(slugs)
rel["reciproca"] = [(d, o) in pares for o, d in zip(rel["slug_origem"], rel["slug_destino"])]
rel = rel[["slug_origem", "termo_origem", "termo_destino", "slug_destino", "destino_existe", "reciproca"]].reset_index(drop=True)

sem_destino = int((~rel["destino_existe"]).sum())
auto = int((rel["slug_origem"] == rel["slug_destino"]).sum())
assim = int((~rel["reciproca"]).sum())
registrar("glossario · relacionados", "Chaves sem correspondência: 'Veja também' apontando para verbete inexistente",
          "Manter na tabela com destino_existe = False (o site oculta o link); corrigir na fonte", f"{sem_destino} de {len(rel)} relações sem destino; {auto} autorreferências", n=sem_destino + auto)
registrar("glossario · relacionados", "Relações assimétricas: A cita B, mas B não cita A",
          "Manter como o autor escreveu (relação dirigida) e criar a coluna derivada `reciproca`; decidir na revisão se o link deve ser de mão dupla",
          f"{assim} de {len(rel)} relações sem retorno ({100 * assim / max(len(rel), 1):.0f}%): " + "; ".join(f"{a} → {b}" for a, b in rel.loc[~rel.reciproca, ["termo_origem", "termo_destino"]].itertuples(index=False)), n=assim)
contar("glossario_relacoes", "após G3 (explodir relacionados)", len(rel))
rel

,slug_origem,termo_origem,termo_destino,slug_destino,destino_existe,reciproca
0,reserva-de-emergencia,Reserva de emergência,Orçamento pessoal,orcamento-pessoal,True,True
1,reserva-de-emergencia,Reserva de emergência,Endividamento,endividamento,True,True
2,reserva-de-emergencia,Reserva de emergência,Juros,juros,True,False
3,orcamento-pessoal,Orçamento pessoal,Reserva de emergência,reserva-de-emergencia,True,True
4,orcamento-pessoal,Orçamento pessoal,Endividamento,endividamento,True,True
5,juros,Juros,Endividamento,endividamento,True,True
6,juros,Juros,Inflação,inflacao,True,True
7,endividamento,Endividamento,Juros,juros,True,True
8,endividamento,Endividamento,Orçamento pessoal,orcamento-pessoal,True,True
9,endividamento,Endividamento,Reserva de emergência,reserva-de-emergencia,True,True


In [9]:
# ---------- G4: variáveis derivadas (para revisão editorial)
vb["n_relacionados"] = vb["relacionados"].map(len)
vb["n_caracteres_definicao"] = vb["definicao"].map(lambda s: 0 if s is pd.NA else len(s))
for campo in ("exemplo", "importa", "fonte"):
    vb[f"tem_{campo}"] = vb[campo].notna()

n = len(vb)
registrar("glossario · fonte", "Campo `fonte` ausente: o verbete não diz de onde veio a definição",
          "Manter ausente (não inventar fonte); marcar `tem_fonte = False` e tratar como pendência de revisão antes de publicar",
          f"{int((~vb.tem_fonte).sum())} de {n} verbetes sem fonte ({100 * (~vb.tem_fonte).mean():.0f}%). Afeta a rastreabilidade do glossário", n=int((~vb.tem_fonte).sum()), status="pendência")
registrar("glossario · exemplo/importa", "Campos opcionais vazios", "Manter; sem imputação",
          f"exemplo ausente em {int((~vb.tem_exemplo).sum())}, 'por que importa' ausente em {int((~vb.tem_importa).sum())} de {n}", n=int((~vb.tem_exemplo).sum() + (~vb.tem_importa).sum()))
curtas, longas = int((vb.n_caracteres_definicao < 40).sum()), int((vb.n_caracteres_definicao > 300).sum())
registrar("glossario · definicao", "Valores extremos de tamanho (definição muito curta ou muito longa)", "Apenas sinalizar (limites 40 e 300 caracteres); não alterar o texto",
          f"{curtas} curtas e {longas} longas; mínimo {int(vb.n_caracteres_definicao.min())}, máximo {int(vb.n_caracteres_definicao.max())} caracteres", n=curtas + longas)
temas = vb["tema"].value_counts()
registrar("glossario · tema", "Temas com poucos verbetes (filtro do site pouco útil)", "Apenas registrar", f"{len(temas)} temas: " + "; ".join(f"{t} ({k})" for t, k in temas.items()))
vb_final = vb[["termo", "slug", "tema", "definicao", "exemplo", "importa", "fonte", "n_relacionados", "n_caracteres_definicao", "tem_exemplo", "tem_importa", "tem_fonte"]].sort_values("slug").reset_index(drop=True)
contar("glossario_verbetes", "após G4 (tabela final)", len(vb_final))
vb_final[["termo", "tema", "n_relacionados", "n_caracteres_definicao", "tem_exemplo", "tem_fonte"]]

,termo,tema,n_relacionados,n_caracteres_definicao,tem_exemplo,tem_fonte
0,Endividamento,Crédito e dívidas,3,102,True,False
1,Inflação,Economia geral,2,95,True,False
2,Juros,Crédito e dívidas,2,114,True,False
3,Orçamento pessoal,Planejamento financeiro,2,118,True,False
4,Reserva de emergência,Planejamento financeiro,3,126,True,False
5,Taxa Selic,Economia geral,2,119,True,False


In [10]:
# ---------- G5: cobertura do questionário pelo glossário (junção analítica por tema; ver docs/questionario.md, seção 7)
EXIGE = {"b1_reserva_def": ["Reserva de emergência"], "b2_reserva_meses": ["Reserva de emergência"], "b3_inflacao": ["Inflação", "Taxa Selic"],
         "b4_juros_compostos": ["Juros", "Endividamento"], "b5_diversificacao": ["Investimentos"], "c1_orcamento": ["Orçamento pessoal"],
         "c2_tem_reserva": ["Reserva de emergência"], "c5_dividas": ["Juros", "Endividamento"], "c6_poupa_regular": ["Orçamento pessoal"]}
cobertura = pd.DataFrame([{"item_questionario": q, "verbete_necessario": v, "slug": slug(v)} for q, vs in EXIGE.items() for v in vs])
cobertura["existe_no_glossario"] = cobertura["slug"].isin(slugs)
n_antes = len(cobertura)
chave_ausente = sorted(cobertura.loc[~cobertura.existe_no_glossario, "verbete_necessario"].unique())
registrar("cobertura questionário × glossário", "Chaves sem correspondência na junção: itens do questionário cujo verbete ainda não existe",
          "Não criar verbete sem fonte para 'fechar' a conta; registrar como pendência de conteúdo (tarefa 12)",
          f"{int((~cobertura.existe_no_glossario).sum())} de {n_antes} pares item-verbete sem correspondência; verbetes ausentes: {', '.join(chave_ausente) or 'nenhum'}", n=int((~cobertura.existe_no_glossario).sum()), status="pendência")
contar("cobertura", "pares item-verbete (antes da junção)", n_antes)
contar("cobertura", "pares item-verbete com verbete existente (depois da junção)", int(cobertura.existe_no_glossario.sum()))
cobertura

,item_questionario,verbete_necessario,slug,existe_no_glossario
0,b1_reserva_def,Reserva de emergência,reserva-de-emergencia,True
1,b2_reserva_meses,Reserva de emergência,reserva-de-emergencia,True
2,b3_inflacao,Inflação,inflacao,True
3,b3_inflacao,Taxa Selic,taxa-selic,True
4,b4_juros_compostos,Juros,juros,True
5,b4_juros_compostos,Endividamento,endividamento,True
6,b5_diversificacao,Investimentos,investimentos,False
7,c1_orcamento,Orçamento pessoal,orcamento-pessoal,True
8,c2_tem_reserva,Reserva de emergência,reserva-de-emergencia,True
9,c5_dividas,Juros,juros,True


### 5.2 Questionário (regras de limpeza)
Etapas: **Q1** envios duplicados · **Q2** consentimento · **Q3** remover identificação · **Q4** normalizar categorias e invalidar códigos · **Q5** consistência lógica · **Q6** ausências · **Q7** escore e níveis · **Q8** tabelas organizadas (tidy).

Com dados sintéticos, os defeitos plantados são: 3 envios duplicados, 4 respostas sem consentimento, colunas de identificação, 6 códigos inválidos, 6 respostas inconsistentes (c3/c4 sem reserva), 4 respostas com mais de 3 temas, ausências, variações de grafia e combinações contraditórias (opções excludentes marcadas junto com outras) que surgem do sorteio.

In [11]:
q = questionario_bruto.copy()
contar("questionario", "bruto", len(q))

# ---------- Q1: envios duplicados (comparando todas as colunas, inclusive o carimbo, ANTES de remover identificação)
n_dup = int(q.duplicated(keep="first").sum())
q = q.drop_duplicates(keep="first").reset_index(drop=True)
contar("questionario", "após Q1 (envios duplicados)", len(q))
registrar("questionario · linhas", "Envios duplicados (linhas idênticas, inclusive o carimbo de data/hora)", "Manter a primeira ocorrência",
          f"{n_dup} removidos; {len(questionario_bruto)} → {len(q)} linhas", n=n_dup)

# ---------- Q2: consentimento (só quem aceitou participa)
q.columns = [str(c).strip().lower() for c in q.columns]
cons = q["consentimento"].map(norm_cod)
sem_cons = int((cons != "aceito").sum())
q = q[cons == "aceito"].reset_index(drop=True)
contar("questionario", "após Q2 (consentimento)", len(q))
registrar("questionario · consentimento", "Respostas de quem não aceitou participar (ou sem registro de aceite)", "Excluir da análise; nenhuma informação delas é usada",
          f"{sem_cons} removidas; {len(q)} linhas restantes", n=sem_cons)

# ---------- Q3: remover identificação e rastreio
q, descartadas = M.limpar_colunas(q)
registrar("questionario · colunas", "Colunas de identificação ou rastreio na exportação", "Descartar antes de qualquer tabela (função `limpar_colunas` do script 02)",
          f"Descartadas: {descartadas or 'nenhuma'}", n=len(descartadas))
testar("Questionário: nenhuma coluna de identificação sobrou", not any(c in q.columns for c in ("nome", "e-mail", "email", "cpf", "matricula", "matrícula", "carimbo de data/hora")))
if "e3_termo_dificil" in q.columns:
    n_txt = int(q["e3_termo_dificil"].map(limpa_texto).notna().sum())
    q = q.drop(columns=["e3_termo_dificil"])
    registrar("questionario · e3_termo_dificil", "Resposta aberta pode conter dado pessoal digitado pelo respondente", "Remover das bases tratadas; revisão manual separada antes de qualquer uso",
              f"{n_txt} respostas abertas preenchidas não entram nas tabelas", n=n_txt)

In [12]:
# ---------- Q4: normalizar categorias e invalidar códigos fora do domínio
q["a1_curso"] = q["a1_curso"].map(norm_cod)
inval, variantes = {}, 0
for v, dominio in DOMINIOS.items():
    if v == "consentimento" or v not in q.columns:
        continue
    if v in MULTIPLAS:
        def limpa_multipla(x, dom=dominio, v=v):
            if x is pd.NA or pd.isna(x): return pd.NA
            itens = [norm_cod(i) for i in str(x).split(";")]
            itens = [i for i in itens if i is not pd.NA]
            validos = list(dict.fromkeys(i for i in itens if i in dom))        # sem repetir, preserva a ordem
            inval[v] = inval.get(v, 0) + sum(1 for i in itens if i not in dom)
            return ";".join(validos) if validos else pd.NA
        antes_m = q[v].copy()
        q[v] = q[v].map(limpa_multipla)
        variantes += int(((antes_m.astype("string") != q[v].astype("string")) & q[v].notna()).sum())
    else:
        novo = q[v].map(norm_cod)
        variantes += int(((q[v].astype("string") != novo.astype("string")) & novo.notna() & novo.isin(dominio.keys())).sum())
        ruim = novo.notna() & ~novo.isin(dominio.keys())
        inval[v] = int(ruim.sum())
        q[v] = novo.where(~ruim, pd.NA)
inval = {k: n for k, n in inval.items() if n}
registrar("questionario · categorias", "Variações de grafia do mesmo código (maiúsculas, acentos, espaços, hífen)", "Normalizar com `norm_cod` (sem acento, minúsculo, '_' no lugar de espaço/hífen)",
          f"{variantes} células de domínio fechado tiveram a grafia normalizada; sem perda de linhas", n=variantes)
registrar("questionario · códigos inválidos", "Valores fora do domínio do livro de códigos", "Transformar em ausente (NA); não adivinhar a intenção do respondente",
          f"{sum(inval.values())} valores invalidados: " + (", ".join(f"{k} ({n})" for k, n in inval.items()) or "nenhum"), n=sum(inval.values()))
for v, dominio in DOMINIOS.items():
    if v == "consentimento" or v not in q.columns or v in MULTIPLAS: continue
    testar(f"Domínio respeitado: {v}", bool(q[v].dropna().isin(dominio.keys()).all()))

In [13]:
# ---------- Q5: consistência lógica
sem_reserva = q["c2_tem_reserva"].eq("nao") & (q["c3_reserva_cobertura"].notna() | q["c4_onde_guarda"].notna())
q.loc[sem_reserva, ["c3_reserva_cobertura", "c4_onde_guarda"]] = pd.NA
registrar("questionario · c3/c4 × c2", "Cobertura e local da reserva preenchidos por quem diz não ter reserva", "Anular c3 e c4 (a resposta a c2 prevalece, pois c3/c4 só valem com reserva)",
          f"{int(sem_reserva.sum())} respostas ajustadas; nenhuma linha removida", n=int(sem_reserva.sum()))

def so_exclusiva(x, exclusivo):
    """Se a opção exclusiva (ex.: 'nenhuma') vier junto de outras, mantém as outras."""
    if x is pd.NA or pd.isna(x): return x
    itens = x.split(";")
    return ";".join(i for i in itens if i != exclusivo) if (exclusivo in itens and len(itens) > 1) else x
for v, excl in (("c5_dividas", "nenhuma"), ("d7_educ_recebida", "nunca")):
    antes = q[v].copy(); q[v] = q[v].map(lambda x, e=excl: so_exclusiva(x, e))
    n_aj = int((antes.fillna("") != q[v].fillna("")).sum())
    registrar(f"questionario · {v}", f"Opção excludente '{excl}' marcada junto com outras", f"Manter apenas as outras opções (contradição resolvida a favor da resposta positiva)", f"{n_aj} respostas ajustadas", n=n_aj)

mais3 = int((q["e1_temas_interesse"].dropna().str.count(";") + 1 > 3).sum())
registrar("questionario · e1_temas_interesse", "Mais de 3 temas marcados (o limite do formulário é 3)", "Manter todos os marcados (a ordem de preferência é desconhecida); sinalizar",
          f"{mais3} respostas acima do limite; recomenda-se limitar no formulário", n=mais3)

In [14]:
# ---------- Q6: ausências (sem imputação)
vars_q = [v for v in DOMINIOS if v != "consentimento" and v in q.columns]
aus = (q[vars_q].isna().mean() * 100).round(1)
condicionais = {"c3_reserva_cobertura", "c4_onde_guarda"}
principais = aus[(aus > 0) & ~aus.index.isin(condicionais)].sort_values(ascending=False)
registrar("questionario · ausências", "Valores ausentes (em branco ou invalidados em Q4)",
          "Não imputar. 'Não sei' e 'prefiro não responder' são categorias válidas; ausente fica fora do denominador de cada pergunta",
          "; ".join(f"{v} {p}%" for v, p in principais.head(8).items()) + "; (c3/c4 ausentes por desenho: só valem com reserva)")
q.insert(0, "id_resposta", range(1, len(q) + 1))
contar("questionario", "após Q6 (base limpa)", len(q))

In [15]:
# ---------- Q7: escore de conhecimento e nível (regras do script 02 / docs/questionario.md)
acertos = pd.DataFrame({c: (q[c] == g).astype(int) for c, g in M.GABARITO.items()})
q["escore_conhecimento"] = acertos.sum(axis=1)
q["nivel_conhecimento"] = q["escore_conhecimento"].map(M.nivel)
q["b1_acertou"] = acertos["b1_reserva_def"].map({1: "acertou", 0: "nao_acertou"})
sem_resp = int(q[list(M.GABARITO)].isna().any(axis=1).sum())
registrar("questionario · escore", "Respondentes sem resposta válida em algum dos 4 itens pontuados",
          "Ausente conta como não-acerto (mesma regra de 'não sei'); b2 e b6 não pontuam",
          f"{sem_resp} de {len(q)} respondentes afetados; escore médio {q.escore_conhecimento.mean():.2f}", n=sem_resp)
display(q["nivel_conhecimento"].value_counts().rename("n").to_frame().join((q["nivel_conhecimento"].value_counts(normalize=True) * 100).round(1).rename("%")))

,n,%
nivel_conhecimento,,
medio,42,75.0
baixo,8,14.3
alto,6,10.7


In [16]:
# ---------- Q8: organização tidy. Observação = respondente (1 linha); múltipla escolha vira tabela longa (1 linha por opção marcada)
respostas = q.copy()
longa = (respostas[["id_resposta"] + sorted(MULTIPLAS)].melt(id_vars="id_resposta", var_name="variavel", value_name="categoria")
         .dropna(subset=["categoria"]).assign(categoria=lambda d: d["categoria"].str.split(";")).explode("categoria")
         .sort_values(["id_resposta", "variavel", "categoria"]).reset_index(drop=True))
contar("questionario_respostas", "tabela final (1 linha por respondente)", len(respostas))
contar("questionario_multipla", "tabela longa (1 linha por opção marcada)", len(longa))
registrar("questionario · múltipla escolha", "Várias opções numa só célula separadas por ';' (viola 'uma variável por coluna')",
          "Manter a forma ampla para o escore e criar a tabela longa `multipla` (uma opção por linha) para contagens",
          f"{len(longa)} opções marcadas por {respostas.id_resposta.nunique()} respondentes em {longa.variavel.nunique()} variáveis")
longa.head()

,id_resposta,variavel,categoria
0,1,c4_onde_guarda,conta_corrente
1,1,c4_onde_guarda,outro
2,1,c5_dividas,atrasadas
3,1,d7_educ_recebida,escola
4,1,d7_educ_recebida,familia


## 6. Testes de integridade e tabelas de conferência

In [17]:
# ---- Glossário
testar("Glossário: hash do arquivo original inalterado (preservação)", hashlib.sha256(GLOSSARIO.read_bytes()).hexdigest() == HASH_GLOSSARIO)
testar("Glossário: contagem preservada (bruto = tratado)", len(vb_final) == len(vb_bruto), f"{len(vb_bruto)} → {len(vb_final)}")
testar("Glossário: 1 verbete por linha, slug único", vb_final["slug"].is_unique)
testar("Glossário: relações = soma de n_relacionados", len(rel) == int(vb_final["n_relacionados"].sum()), f"{len(rel)} relações")
testar("Glossário: tipos esperados", vb_final["tem_fonte"].dtype == bool and vb_final["n_caracteres_definicao"].dtype.kind in "iu")
testar("Junção cobertura: contagem antes = linhas da tabela", len(cobertura) == n_antes)

# ---- Questionário
f = pd.DataFrame(funil).query("base == 'questionario'").set_index("etapa")["linhas"]
testar("Questionário: funil fecha (bruto − duplicados − sem consentimento = base limpa)",
       f["bruto"] - n_dup - sem_cons == f["após Q6 (base limpa)"], f"{f['bruto']} − {n_dup} − {sem_cons} = {f['após Q6 (base limpa)']}")
testar("Questionário: id_resposta único e sequencial", respostas["id_resposta"].tolist() == list(range(1, len(respostas) + 1)))
testar("Questionário: escore entre 0 e 4", bool(respostas["escore_conhecimento"].between(0, 4).all()))
testar("Questionário: nível definido para todos", bool(respostas["nivel_conhecimento"].notna().all()))
testar("Questionário: tabela longa só tem categorias válidas", all(c in DOMINIOS[v] for v, c in zip(longa["variavel"], longa["categoria"])))
testar("Questionário: ninguém sem reserva tem c3/c4", not bool((respostas["c2_tem_reserva"].eq("nao") & (respostas["c3_reserva_cobertura"].notna() | respostas["c4_onde_guarda"].notna())).any()))
testar("Questionário: 'nenhuma' não aparece junto de outras dívidas", not bool(respostas["c5_dividas"].dropna().map(lambda s: "nenhuma" in s.split(";") and len(s.split(";")) > 1).any()))
if ORIGEM_Q == "SINTETICO":      # só faz sentido conferir contra os defeitos plantados quando a base é a sintética
    testar("Teste sintético: 3 duplicatas plantadas foram removidas", n_dup == 3, f"removidas {n_dup}")
    testar("Teste sintético: 4 sem consentimento removidos", sem_cons == 4, f"removidos {sem_cons}")
    testar("Teste sintético: 6 códigos inválidos detectados", sum(inval.values()) >= 6, f"detectados {sum(inval.values())}")
    testar("Teste sintético: colunas de identificação descartadas", set(descartadas) >= {"carimbo de data/hora", "nome"}, str(descartadas))

# ---- Tabelas de conferência
display(pd.DataFrame(funil).pivot_table(index=["base", "etapa"], values="linhas", sort=False))
pd.DataFrame(testes)

linhas
base                   etapa                                                     
glossario              bruto                                                  6.0
                       após G2 (duplicidades)                                 6.0
glossario_relacoes     após G3 (explodir relacionados)                       14.0
glossario_verbetes     após G4 (tabela final)                                 6.0
cobertura              pares item-verbete (antes da junção)                  12.0
                       pares item-verbete com verbete existente (depoi...    11.0
questionario           bruto                                                 63.0
                       após Q1 (envios duplicados)                           60.0
                       após Q2 (consentimento)                               56.0
                       após Q6 (base limpa)                                  56.0
questionario_respostas tabela final (1 linha por respondente)                56.0
questionario_multipla  tabela longa (1 linha por opção marcada)             320.0

,teste,resultado,detalhe
0,Glossário: chave (slug) única e não vazia,OK,
1,Questionário: nenhuma coluna de identificação ...,OK,
2,Domínio respeitado: a2_periodo,OK,
3,Domínio respeitado: a3_idade,OK,
4,Domínio respeitado: a4_ocupacao,OK,
5,Domínio respeitado: a5_renda_propria,OK,
6,Domínio respeitado: b1_reserva_def,OK,
7,Domínio respeitado: b2_reserva_meses,OK,
8,Domínio respeitado: b3_inflacao,OK,
9,Domínio respeitado: b4_juros_compostos,OK,


## 7. Exportação e resumo final das decisões

In [18]:
# ---- Tabelas agregadas do questionário (mesmas funções do script 02; células < 5 suprimidas)
ignorar = {"id_resposta", "escore_conhecimento"}
variaveis = [c for c in respostas.columns if c not in ignorar]
freq = M.frequencias(respostas, variaveis, MULTIPLAS)
dist = respostas["escore_conhecimento"].value_counts().sort_index()
dist = pd.DataFrame({"escore": dist.index, "n": [M.suprimir(n) for n in dist.values], "base": len(respostas)})
cruz = pd.concat([M.cruzar(respostas, a, b) for a, b in M.CRUZAMENTOS], ignore_index=True)
med = pd.concat([M.medias(respostas, g) for g in M.MEDIAS_POR], ignore_index=True)
testar("Frequências: 'base' de cada variável = respostas válidas", all(int(r.base) == int(respostas[r.variavel].notna().sum()) for r in freq.itertuples() if r.variavel in respostas.columns))
testar("Frequências: nenhuma célula com n < 5 exposta", all((x == "<5") or int(x) >= MIN_CELULA for x in freq["n"]))

destino_q = PROCESSADOS if ORIGEM_Q == "REAL" else SINTETICO      # sintético NUNCA entra em data/processed
for nome, tabela in (("frequencias", freq), ("escore_distribuicao", dist), ("cruzamentos", cruz), ("escore_medio_por_grupo", med)):
    tabela.to_csv(destino_q / f"{nome}.csv", index=False)
if ORIGEM_Q == "SINTETICO":
    respostas.to_csv(SINTETICO / "respostas_sinteticas_limpas.csv", index=False)
    longa.to_csv(SINTETICO / "respostas_sinteticas_multipla.csv", index=False)
    (SINTETICO / "LEIA-ME.md").write_text(
        "# Teste com dados SINTÉTICOS\n\n"
        "Tudo nesta pasta foi gerado pelo notebook 03_obtencao_limpeza_tratamento.ipynb a partir de uma base inventada "
        "(semente fixa) com defeitos plantados, só para provar que as regras de limpeza funcionam.\n\n"
        "**Não é resultado de pesquisa e não descreve nenhuma pessoa.** Não cite estes números. "
        "Quando existir data/raw/respostas_questionario.csv, o notebook trata os dados reais e grava as tabelas agregadas em data/processed/.\n",
        encoding="utf-8")

# ---- Glossário (dado real) e documentação do processo
vb_final.to_csv(PROCESSADOS / "glossario_verbetes.csv", index=False)
rel.to_csv(PROCESSADOS / "glossario_relacoes.csv", index=False)
cobertura.to_csv(PROCESSADOS / "glossario_cobertura_questionario.csv", index=False)
livro.to_csv(SAIDAS / "livro_de_codigos.csv", index=False)
pd.DataFrame(relatorio).to_csv(SAIDAS / "relatorio_qualidade.csv", index=False)
pd.DataFrame(funil).to_csv(SAIDAS / "funil_contagens.csv", index=False)
pd.DataFrame(testes).to_csv(SAIDAS / "testes_integridade.csv", index=False)
pd.DataFrame(aquisicao).to_csv(SAIDAS / "registro_aquisicao.csv", index=False)

print(f"Questionário tratado a partir de dados {ORIGEM_Q}. Tabelas agregadas gravadas em: {destino_q.relative_to(RAIZ)}/")
print(f"Testes: {sum(t['resultado'] == 'OK' for t in testes)} OK de {len(testes)}")
print("Glossário tratado gravado em data/processed/ (3 tabelas). Relatório de qualidade: outputs/relatorio_qualidade.csv")

Questionário tratado a partir de dados SINTETICO. Tabelas agregadas gravadas em: outputs/sintetico_teste/
Testes: 42 OK de 42
Glossário tratado gravado em data/processed/ (3 tabelas). Relatório de qualidade: outputs/relatorio_qualidade.csv


In [19]:
# ---- Relatório de qualidade (Variável/base · Problema · Decisão · Impacto/validação)
pd.set_option("display.max_colwidth", 200)
pd.DataFrame(relatorio).rename(columns={"base_variavel": "Variável/base", "problema_encontrado": "Problema encontrado",
                                         "decisao_aplicada": "Decisão aplicada", "impacto_validacao": "Impacto/validação"})

,Variável/base,status,ocorrencias,Problema encontrado,Decisão aplicada,Impacto/validação
0,glossario · textos,sem ocorrência,0.0,Espaços extras ou formas Unicode diferentes nos campos de texto,"Normalizar para NFC, colapsar espaços e aparar as pontas",0 células alteradas de 36; nenhuma linha removida
1,glossario · termo/slug,sem ocorrência,0.0,Termos duplicados (mesmo identificador no site) ou sem nome,Nenhuma ação necessária,0 duplicados e 0 sem termo em 6 verbetes
2,glossario · relacionados,sem ocorrência,0.0,Chaves sem correspondência: 'Veja também' apontando para verbete inexistente,Manter na tabela com destino_existe = False (o site oculta o link); corrigir na fonte,0 de 14 relações sem destino; 0 autorreferências
3,glossario · relacionados,tratado,2.0,"Relações assimétricas: A cita B, mas B não cita A",Manter como o autor escreveu (relação dirigida) e criar a coluna derivada `reciproca`; decidir na revisão se o link deve ser de mão dupla,2 de 14 relações sem retorno (14%): Reserva de emergência → Juros; Taxa Selic → Juros
4,glossario · fonte,pendência,6.0,Campo `fonte` ausente: o verbete não diz de onde veio a definição,Manter ausente (não inventar fonte); marcar `tem_fonte = False` e tratar como pendência de revisão antes de publicar,6 de 6 verbetes sem fonte (100%). Afeta a rastreabilidade do glossário
5,glossario · exemplo/importa,sem ocorrência,0.0,Campos opcionais vazios,Manter; sem imputação,"exemplo ausente em 0, 'por que importa' ausente em 0 de 6"
6,glossario · definicao,sem ocorrência,0.0,Valores extremos de tamanho (definição muito curta ou muito longa),Apenas sinalizar (limites 40 e 300 caracteres); não alterar o texto,"0 curtas e 0 longas; mínimo 95, máximo 126 caracteres"
7,glossario · tema,informativo,NaN,Temas com poucos verbetes (filtro do site pouco útil),Apenas registrar,3 temas: Planejamento financeiro (2); Crédito e dívidas (2); Economia geral (2)
8,cobertura questionário × glossário,pendência,1.0,Chaves sem correspondência na junção: itens do questionário cujo verbete ainda não existe,Não criar verbete sem fonte para 'fechar' a conta; registrar como pendência de conteúdo (tarefa 12),1 de 12 pares item-verbete sem correspondência; verbetes ausentes: Investimentos
9,questionario · linhas,tratado,3.0,"Envios duplicados (linhas idênticas, inclusive o carimbo de data/hora)",Manter a primeira ocorrência,3 removidos; 63 → 60 linhas


### Resumo das decisões e limitações que permanecem

1. **Questionário sem dado real.** O que foi validado são as **regras de limpeza**, não resultados. Ao exportar as respostas reais para `data/raw/respostas_questionario.csv`, reexecute o notebook: ele troca a base sintética pela real e grava as tabelas agregadas em `data/processed/`. Se o formulário exportar códigos diferentes de `outputs/livro_de_codigos.csv`, ajuste o formulário ou `DOMINIOS`.
2. **Glossário com 6 verbetes de rascunho e nenhum com fonte.** É dado real, mas pequeno e incompleto; a falta de `fonte` e de verbetes exigidos pelo questionário (ver a tabela de cobertura) é pendência de conteúdo, não de código.
3. **Sem imputação.** Ausentes e inválidos ficam como NA; "não sei" e "prefiro não responder" são categorias válidas.
4. **Privacidade.** Respostas individuais, identificação e texto livre nunca entram nas tabelas versionadas. Células com menos de 5 respondentes aparecem como `<5`.
5. **Limites do desenho.** Amostra por conveniência, autodeclaração e relato do estudante sobre a família (ver `docs/questionario.md`).